# 02 Text Features + EDA

Fetch earnings-call transcripts from Hugging Face, normalize dates/keys, create lightweight text features, and export a text feature table for downstream merging.

In [ ]:
from pathlib import Path
import math
import re
import string
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TEXT_FEATURES_OUTPUT = PROCESSED_DIR / "earnings_text_features.csv"
TEXT_RAW_OUTPUT = PROCESSED_DIR / "earnings_transcripts_hf_raw.csv"

HF_DATASET = "TQTfintech/earnings-transcripts"
HF_ROWS_URL = "https://datasets-server.huggingface.co/rows"
HF_DATASET

In [ ]:
def fetch_hf_rows(dataset: str, split: str = "train", page_size: int = 100) -> pd.DataFrame:
    rows = []
    offset = 0
    while True:
        params = {
            "dataset": dataset,
            "config": "default",
            "split": split,
            "offset": offset,
            "length": page_size,
        }
        response = requests.get(HF_ROWS_URL, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()
        page_rows = [item["row"] for item in payload.get("rows", [])]
        rows.extend(page_rows)
        if len(page_rows) < page_size:
            break
        offset += page_size
    return pd.DataFrame(rows)

transcripts_raw = fetch_hf_rows(HF_DATASET)
transcripts_raw.to_csv(TEXT_RAW_OUTPUT, index=False)

transcripts_raw.shape, transcripts_raw.head(3)

In [ ]:
MONTH_PATTERN = r"Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?"
DATE_RE = re.compile(
    rf"(?:Mon(?:day)?|Tue(?:sday)?|Wed(?:nesday)?|Thu(?:rsday)?|Fri(?:day)?|Sat(?:urday)?|Sun(?:day)?)?,?\s*((?:{MONTH_PATTERN})\.?\s+\d{{1,2}},\s+\d{{4}})",
    re.IGNORECASE,
)
SENTENCE_RE = re.compile(r"[.!?]+")
TOKEN_RE = re.compile(r"\b[a-zA-Z][a-zA-Z'\-]*\b")
TICKER_ALIASES = {"GOOGL": "GOOG"}

POSITIVE_TERMS = {
    "growth", "grew", "strong", "strength", "improve", "improved", "improvement",
    "profit", "profitable", "beat", "upside", "accelerate", "accelerated", "record",
    "momentum", "demand", "opportunity", "confident", "resilient", "expand", "expanded",
}
NEGATIVE_TERMS = {
    "risk", "risks", "decline", "declined", "weak", "weakness", "loss", "losses",
    "headwind", "headwinds", "constraint", "constraints", "pressure", "pressures",
    "volatility", "volatile", "slowdown", "uncertain", "uncertainty", "down", "decrease",
}
FINANCE_TERMS = {
    "revenue", "margin", "guidance", "earnings", "cash", "growth", "customers", "demand",
    "segment", "operating", "income", "expense", "capital", "share", "market", "sales",
}


def parse_transcript_date(text: str) -> pd.Timestamp:
    match = DATE_RE.search(str(text).replace("\n", " "))
    if match is None:
        return pd.NaT
    date_text = match.group(1).replace(".", "")
    return pd.to_datetime(date_text, errors="coerce")


def clean_text(text: str) -> str:
    text = str(text).replace("\u00a0", " ")
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def text_features(text: str) -> pd.Series:
    cleaned = clean_text(text)
    tokens = TOKEN_RE.findall(cleaned)
    sentences = [part.strip() for part in SENTENCE_RE.split(cleaned) if part.strip()]
    token_count = len(tokens)
    sentence_count = len(sentences)
    unique_count = len(set(tokens))
    pos_count = sum(token in POSITIVE_TERMS for token in tokens)
    neg_count = sum(token in NEGATIVE_TERMS for token in tokens)
    finance_count = sum(token in FINANCE_TERMS for token in tokens)
    avg_word_length = np.mean([len(token) for token in tokens]) if tokens else np.nan

    return pd.Series(
        {
            "transcript_chars": len(cleaned),
            "word_count": token_count,
            "sentence_count": sentence_count,
            "avg_sentence_length": token_count / sentence_count if sentence_count else np.nan,
            "avg_word_length": avg_word_length,
            "unique_word_ratio": unique_count / token_count if token_count else np.nan,
            "positive_term_count": pos_count,
            "negative_term_count": neg_count,
            "finance_term_count": finance_count,
            "positive_term_rate": pos_count / token_count if token_count else np.nan,
            "negative_term_rate": neg_count / token_count if token_count else np.nan,
            "finance_term_rate": finance_count / token_count if token_count else np.nan,
            "simple_sentiment_balance": (pos_count - neg_count) / token_count if token_count else np.nan,
            "clean_transcript": cleaned,
        }
    )

In [ ]:
transcripts = transcripts_raw.copy()
transcripts["ticker"] = transcripts["ticker"].astype(str).str.upper().str.strip().replace(TICKER_ALIASES)
transcripts["call_date_dt"] = transcripts["transcript"].apply(parse_transcript_date)
transcripts["call_date"] = transcripts["call_date_dt"].dt.strftime("%Y-%m-%d")

feature_frame = transcripts["transcript"].apply(text_features)
text_features_df = pd.concat(
    [
        transcripts[["ticker", "company", "quarter", "year", "period", "call_date", "call_date_dt", "source_url", "scraped_date"]],
        feature_frame,
    ],
    axis=1,
)
text_features_df["text_call_key"] = text_features_df["ticker"] + "_" + text_features_df["call_date"].astype(str).str.replace("-", "_", regex=False)

text_qa = pd.DataFrame(
    {
        "metric": ["rows", "tickers", "missing_call_date", "duplicate_text_keys", "date_min", "date_max"],
        "value": [
            len(text_features_df),
            text_features_df["ticker"].nunique(),
            int(text_features_df["call_date_dt"].isna().sum()),
            int(text_features_df.loc[text_features_df["call_date_dt"].notna(), "text_call_key"].duplicated().sum()),
            text_features_df["call_date_dt"].min(),
            text_features_df["call_date_dt"].max(),
        ],
    }
)
text_qa

In [ ]:
export_cols = [
    "text_call_key",
    "ticker",
    "company",
    "quarter",
    "year",
    "period",
    "call_date",
    "source_url",
    "scraped_date",
    "transcript_chars",
    "word_count",
    "sentence_count",
    "avg_sentence_length",
    "avg_word_length",
    "unique_word_ratio",
    "positive_term_count",
    "negative_term_count",
    "finance_term_count",
    "positive_term_rate",
    "negative_term_rate",
    "finance_term_rate",
    "simple_sentiment_balance",
    "clean_transcript",
]

text_features_export = text_features_df[export_cols].sort_values(["ticker", "call_date"]).reset_index(drop=True)
assert text_features_export["ticker"].notna().all(), "Ticker normalization failed."
assert text_features_export["word_count"].gt(0).all(), "At least one transcript has no text tokens."

text_features_export.to_csv(TEXT_FEATURES_OUTPUT, index=False)
print(f"Wrote text features: {TEXT_FEATURES_OUTPUT}")
text_features_export.head()

In [ ]:
text_summary_cols = [
    "transcript_chars",
    "word_count",
    "sentence_count",
    "avg_sentence_length",
    "unique_word_ratio",
    "positive_term_rate",
    "negative_term_rate",
    "finance_term_rate",
    "simple_sentiment_balance",
]

text_features_export[text_summary_cols].describe().T

In [ ]:
ticker_text_profile = (
    text_features_export.groupby("ticker", as_index=False)
    .agg(
        transcripts=("text_call_key", "count"),
        avg_word_count=("word_count", "mean"),
        avg_sentiment_balance=("simple_sentiment_balance", "mean"),
        avg_negative_rate=("negative_term_rate", "mean"),
        avg_finance_rate=("finance_term_rate", "mean"),
    )
    .sort_values(["transcripts", "ticker"], ascending=[False, True])
)

ticker_text_profile.head(25)

In [ ]:
try:
    import matplotlib.pyplot as plt

    axes = text_features_export[text_summary_cols].hist(figsize=(14, 10), bins=30)
    plt.suptitle("Transcript feature distributions", y=1.02)
    plt.tight_layout()
except ImportError:
    print("matplotlib is not installed; skipping histogram plots.")